In [1]:
# ==============================================================
# 1) Setup & imports — Vectorization of monthly GEE mosaics
#    (MEM fix for GDAL/Rasterio included)
# ==============================================================

# If needed in your ArcGIS Pro conda kernel:
# !pip install -q rasterio geopandas shapely pyproj numpy pandas tqdm

from pathlib import Path
import re
from typing import Dict, List, Tuple, Optional

import os
import numpy as np
import pandas as pd
from tqdm import tqdm

import rasterio
from rasterio.mask import mask as rio_mask
from rasterio.features import shapes as rio_shapes
from rasterio.env import Env

import geopandas as gpd
from shapely.geometry import shape as shp_shape, mapping
from shapely.ops import unary_union, transform as shp_transform
from shapely import geometry as sgeom

from pyproj import CRS, Transformer, Geod
from contextlib import contextmanager

# --- Enable GDAL MEM 'DATAPOINTER' open (required by rasterio.mask/shapes on newer GDAL)
os.environ["GDAL_MEM_ENABLE_OPEN"] = "YES"

@contextmanager
def mem_ok():
    """Context manager: ensure GDAL MEM open is allowed for operations that need it."""
    with Env(GDAL_MEM_ENABLE_OPEN='YES'):
        yield

In [2]:
# ==============================================================
# 2) User settings — paths, options, outputs  (UPDATED)
# ==============================================================

# --- INPUTS ---
MOSAIC_DIR = Path(r"C:\Users\ibana\Desktop\JRC_Tanganica\GIS_Intermediate\Intermediate_files\SurfaceWater\GEE_SW\mosaics")
AOI_DIR    = Path(r"C:\Users\ibana\Desktop\JRC_Tanganica\GIS_Intermediate\Intermediate_files\SurfaceWater\aoi")

# HydroLAKES AOI shapefiles
AOI_FILES = {
    "Tanganyika": AOI_DIR / "HydroLAKES_polys_v10_Tanganyika.shp",
    "Kivu":       AOI_DIR / "HydroLAKES_polys_v10_Kivu.shp",
}

# --- OUTPUTS ---
VECTORS_ROOT = Path(r"C:\Users\ibana\Desktop\JRC_Tanganica\GIS_Intermediate\Intermediate_files\SurfaceWater\vectors")
OUT_SUBDIR   = VECTORS_ROOT / "GEE_SW"
(OUT_SUBDIR / "Tanganyika").mkdir(parents=True, exist_ok=True)
(OUT_SUBDIR / "Kivu").mkdir(parents=True, exist_ok=True)

# --- PROCESSING SWITCHES ---
MODE = "BY_LAKE"            # "BY_LAKE" (clip by HydroLAKES+buffer) or "FULL"
BUFFER_KM = 5.0             # buffer around AOIs (km)
OVERWRITE = False
OUTPUT_FMT = "SHP"          # "GPKG" (recommended) or "SHP"
DISSOLVE_TO_SINGLE = False  # dissolve all parts into one multipart
MIN_POLY_AREA_KM2 = 2000    # coarse filter; consider lowering and using overlap filter below
CONNECTIVITY = 8            # 4 or 8
ALL_TOUCHED = False
DIAG_VERBOSE = True
    # NEW: fill small interior holes in polygons (keeps larger holes = real islands)
FILL_SMALL_HOLES = True         # default ON
MAX_HOLE_AREA_KM2     = 1.0          # remove interior holes smaller than this area (km²)

# Single-run testing
RUN_ONLY_YYYY_MM = None   # e.g., "2024_07" or None

# NEW: Overlap-based lake selection (uses UNBUFFERED HydroLAKES polygon)
KEEP_ONLY_AOI_OVERLAP   = True     # filter components by overlap with true lake polygon
AOI_OVERLAP_MIN_PCT     = 70.0     # keep polygons with >= this % of their own area overlapping the lake
KEEP_TOP_N_BY_OVERLAP   = 1        # after filtering, keep only top-N by overlap area (None to keep all)

# Output filename template
OUT_NAME_TMPL = "sw_{YYYY}_{MM}_{LAKE}"

# Geodesic calculator
GEOD = Geod(ellps="WGS84")

# --- PROCESSING SWITCHES ---
# (…your existing settings here…)





In [3]:
# ==============================================================
# 3) Helpers — parse dates, local equal-area buffering, geodesic area
# ==============================================================

def parse_yyyymm_from_name(path: Path) -> Tuple[Optional[int], Optional[int]]:
    """
    Expect: sw_YYYY_MM.tif  → returns (YYYY, MM)
    """
    m = re.match(r"sw_(\d{4})_(\d{2})\.tif$", path.name, flags=re.IGNORECASE)
    if not m:
        return None, None
    return int(m.group(1)), int(m.group(2))

def local_laea_crs(geom: sgeom.base.BaseGeometry) -> CRS:
    """Local Lambert Azimuthal Equal-Area, centered at geometry centroid (meters)."""
    c = geom.centroid
    return CRS.from_proj4(f"+proj=laea +lat_0={c.y} +lon_0={c.x} +x_0=0 +y_0=0 +datum=WGS84 +units=m +no_defs")

def buffer_km_geodesic(aoi_wgs84: sgeom.base.BaseGeometry, buffer_km: float) -> sgeom.base.BaseGeometry:
    """
    Buffer AOI by buffer_km using local LAEA to preserve metric distances, then back to WGS84.
    """
    if buffer_km <= 0:
        return aoi_wgs84

    laea = local_laea_crs(aoi_wgs84)
    to_laea = Transformer.from_crs("EPSG:4326", laea, always_xy=True).transform
    to_wgs  = Transformer.from_crs(laea, "EPSG:4326", always_xy=True).transform

    aoi_laea = shp_transform(to_laea, aoi_wgs84)
    aoi_buf  = aoi_laea.buffer(buffer_km * 1000.0)
    return shp_transform(to_wgs, aoi_buf)

def geodesic_area_km2(geom_wgs84: sgeom.base.BaseGeometry) -> float:
    """
    Geodesic area on WGS84 ellipsoid (km²). Works for Polygon/MultiPolygon.
    """
    if geom_wgs84.is_empty:
        return 0.0

    def ring_area(coords):
        lons, lats = zip(*coords)
        area_m2, _ = GEOD.polygon_area_perimeter(lons, lats)
        return abs(area_m2) / 1e6  # m² → km²

    if isinstance(geom_wgs84, sgeom.Polygon):
        A = ring_area(list(geom_wgs84.exterior.coords))
        for hole in geom_wgs84.interiors:
            A -= ring_area(list(hole.coords))
        return max(0.0, A)

    if isinstance(geom_wgs84, sgeom.MultiPolygon):
        return sum(geodesic_area_km2(p) for p in geom_wgs84.geoms)

    # Fallback: take polygonal part
    poly = geom_wgs84.buffer(0)
    if isinstance(poly, (sgeom.Polygon, sgeom.MultiPolygon)):
        return geodesic_area_km2(poly)
    return 0.0


In [4]:
# ==============================================================
# 4) AOIs — load HydroLAKES, union, buffer (+5 km), WGS84
# ==============================================================

def load_aoi_union(aoi_path: Path) -> sgeom.base.BaseGeometry:
    gdf = gpd.read_file(aoi_path).to_crs("EPSG:4326")
    geom = unary_union(gdf.geometry)
    # normalize & fix minor invalidities
    geom = shp_shape(mapping(geom))
    geom = geom.buffer(0)
    return geom

AOIs_WGS84: Dict[str, sgeom.base.BaseGeometry] = {}
AOIs_BUFFERED: Dict[str, sgeom.base.BaseGeometry] = {}
for lake, p in AOI_FILES.items():
    base = load_aoi_union(p)
    AOIs_WGS84[lake] = base
    AOIs_BUFFERED[lake] = buffer_km_geodesic(base, BUFFER_KM)

print("Prepared buffered AOIs (km²):",
      {k: f"{geodesic_area_km2(v):.1f}" for k, v in AOIs_BUFFERED.items()})


Prepared buffered AOIs (km²): {'Tanganyika': '41200.9', 'Kivu': '4634.2'}


In [5]:
# ==============================================================
# 5) Core helpers — binarize, polygonize, overlap utilities
#     (UPDATED: remove_small_holes helper)
# ==============================================================

def raster_to_binary_water(arr_masked: np.ma.MaskedArray) -> np.ndarray:
    valid = ~arr_masked.mask
    data  = arr_masked.filled(0)
    water = valid & (data != 0)
    return water.astype(np.uint8)

def _to_2d_uint8(arr: np.ndarray, ctx: str = "") -> Optional[np.ndarray]:
    a = np.asarray(arr)
    if a.ndim == 3 and a.shape[0] == 1:
        a = a[0]
    elif a.ndim != 2:
        a = np.squeeze(a)
    if a.ndim != 2 or a.size == 0 or a.shape[0] == 0 or a.shape[1] == 0:
        if DIAG_VERBOSE:
            print(f"[polygonize] Non-2D/empty in {ctx}: shape={getattr(a,'shape',None)} → empty output.")
        return None
    return np.ascontiguousarray((a != 0).astype(np.uint8, copy=False))

def polygonize_water(mask_arr: np.ndarray, transform, connectivity=8, drop_small_km2=0.0,
                     ctx: str = "") -> List[sgeom.base.BaseGeometry]:
    arr2d = _to_2d_uint8(mask_arr, ctx=ctx)
    if arr2d is None:
        return []
    geoms = []
    with mem_ok():
        for geom, value in rio_shapes(arr2d, transform=transform, connectivity=CONNECTIVITY):
            if value != 1:
                continue
            poly = shp_shape(geom)  # WGS84
            if drop_small_km2 > 0.0 and geodesic_area_km2(poly) < drop_small_km2:
                continue
            geoms.append(poly)
    return geoms

# --- NEW: remove tiny interior holes (keep islands above threshold) ---
def remove_small_holes(geom: sgeom.base.BaseGeometry, max_hole_km2: float) -> sgeom.base.BaseGeometry:
    """
    Return geometry with interior holes < max_hole_km2 removed.
    Works for Polygon and MultiPolygon (in WGS84; uses geodesic area).
    """
    if not geom or geom.is_empty or max_hole_km2 <= 0:
        return geom

    def _process_poly(poly: sgeom.Polygon) -> sgeom.Polygon:
        keep_holes = []
        for ring in poly.interiors:
            hole_poly = sgeom.Polygon(ring)
            if geodesic_area_km2(hole_poly) >= max_hole_km2:
                keep_holes.append(ring)  # keep big holes → real islands
        return sgeom.Polygon(poly.exterior, holes=keep_holes)

    if isinstance(geom, sgeom.Polygon):
        return _process_poly(geom).buffer(0)

    if isinstance(geom, sgeom.MultiPolygon):
        parts = [_process_poly(p) for p in geom.geoms]
        return sgeom.MultiPolygon(parts).buffer(0)

    # Fallback: return unchanged
    return geom


In [6]:
# ==============================================================
# 6) Per-mosaic processor — clip (optional), polygonize, FILTER BY LAKE OVERLAP,
#    remove small interior holes (optional), dissolve, and save  [FULL BLOCK]
#    (MEM-safe, robust masked-array handling, SHP-safe field names)
# ==============================================================

def process_mosaic_for_target(mosaic_path: Path,
                              lake_name: Optional[str],
                              aoi_geom_wgs84: Optional[sgeom.base.BaseGeometry],
                              out_dir: Path,
                              overwrite=False,
                              dissolve=True,
                              drop_small_km2=0.0) -> Optional[Path]:
    """
    Vectorize water polygons for either a lake AOI (clipped) or full raster.
    Returns output file path if written, else None.
    """
    yyyy, mm = parse_yyyymm_from_name(mosaic_path)
    if yyyy is None:
        print(f"[skip] Unrecognized mosaic name: {mosaic_path.name}")
        return None

    # Output path & driver + SHP-safe column names
    name_bits = dict(YYYY=f"{yyyy:04d}", MM=f"{mm:02d}", LAKE=(lake_name or "full"))
    out_stem = OUT_NAME_TMPL.format(**name_bits)
    if OUTPUT_FMT.upper() == "SHP":
        out_path = out_dir / f"{out_stem}.shp"; drv = "ESRI Shapefile"
        COL_AREA = "area_km2"   # 8 chars
        COL_OVA  = "ov_km2"     # overlap area (km²)
        COL_OVP  = "ov_pct"     # overlap percent
    else:
        out_path = out_dir / f"{out_stem}.gpkg"; drv = "GPKG"
        COL_AREA = "area_km2"
        COL_OVA  = "overlap_km2"
        COL_OVP  = "overlap_pct"

    if out_path.exists() and not overwrite:
        return out_path

    # Helper to write empty outputs (note: SHP can't store empty geometry → fallback to GPKG)
    def _write_empty():
        gdf_empty = gpd.GeoDataFrame(
            {"year":[yyyy], "month":[mm], "lake":[lake_name or "full"], COL_AREA:[0.0]},
            geometry=[sgeom.MultiPolygon([])], crs="EPSG:4326"
        )
        out_dir.mkdir(parents=True, exist_ok=True)
        if OUTPUT_FMT.upper() == "SHP":
            gdf_empty.to_file(out_path.with_suffix(".gpkg"), driver="GPKG")
        else:
            gdf_empty.to_file(out_path, driver=drv)
        return out_path

    with rasterio.open(mosaic_path) as ds:
        if aoi_geom_wgs84 is not None:
            # Clip to AOI for speed; IMPORTANT: indexes=[1] forces (1,H,W) array
            try:
                with mem_ok():
                    clipped, clipped_transform = rio_mask(
                        ds, [mapping(aoi_geom_wgs84)],
                        crop=True, filled=False, indexes=[1], all_touched=ALL_TOUCHED
                    )
            except ValueError as e:
                if "do not overlap" in str(e).lower():
                    return _write_empty()
                raise

            # Masked (1,H,W) → (H,W)
            arr_m = clipped[0]
            if arr_m.ndim != 2:
                return _write_empty()
            H, W = arr_m.shape

            # Merge value-based nodata into mask if present
            if ds.nodata is not None and not np.isnan(ds.nodata):
                nodata_mask = (arr_m.data == ds.nodata)
                arr_m = np.ma.masked_array(arr_m.data, mask=(arr_m.mask | nodata_mask))

            bin_arr  = raster_to_binary_water(arr_m)
            transform = clipped_transform
            ctx = f"{mosaic_path.name} / {lake_name}"
        else:
            arr_m = ds.read(1, masked=True)  # (H,W) masked
            if arr_m.ndim != 2:
                return _write_empty()
            H, W  = arr_m.shape
            if ds.nodata is not None and not np.isnan(ds.nodata):
                arr_m = np.ma.masked_equal(arr_m, ds.nodata)
            bin_arr  = raster_to_binary_water(arr_m)
            transform = ds.transform
            ctx = f"{mosaic_path.name} / full"

        # Guard: enforce 2-D shape in case of unexpected flattening
        bin_arr = np.asarray(bin_arr)
        if bin_arr.ndim == 1 and bin_arr.size == H * W:
            if DIAG_VERBOSE:
                print(f"[reshape] 1-D → 2-D for {ctx}: ({bin_arr.size},) → ({H},{W})")
            bin_arr = bin_arr.reshape((H, W))
        elif bin_arr.ndim != 2:
            if DIAG_VERBOSE:
                print(f"[polygonize] Non-2D persists in {ctx}: shape={getattr(bin_arr,'shape',None)} → empty.")
            return _write_empty()

        if DIAG_VERBOSE:
            wpx = int((bin_arr != 0).sum())
            print(f"[diag] {ctx}: water_px={wpx}")

        # Polygonize
        polys = polygonize_water(bin_arr, transform,
                                 connectivity=CONNECTIVITY, drop_small_km2=drop_small_km2,
                                 ctx=ctx)

        if not polys:
            return _write_empty()

        # -------- NEW: remove tiny interior holes to suppress omissions ----------
        if FILL_SMALL_HOLES and MAX_HOLE_AREA_KM2 > 0:
            polys = [remove_small_holes(g, MAX_HOLE_AREA_KM2) for g in polys if (g and not g.is_empty)]
            polys = [g for g in polys if (g and not g.is_empty)]
            if not polys:
                return _write_empty()
        # -----------------------------------------------------------------------

        # -------- Overlap filter with UNBUFFERED lake polygon (if BY_LAKE) --------
        if MODE.upper() == "BY_LAKE" and lake_name and KEEP_ONLY_AOI_OVERLAP:
            lake_unbuf = AOIs_WGS84[lake_name]
            kept_polys = []
            kept_attrs = []
            for g in polys:
                A_poly = geodesic_area_km2(g)
                inter = g.intersection(lake_unbuf)
                A_int = geodesic_area_km2(inter) if not inter.is_empty else 0.0
                pct   = (100.0 * A_int / A_poly) if A_poly > 0 else 0.0
                if pct >= AOI_OVERLAP_MIN_PCT:
                    kept_polys.append(g)
                    kept_attrs.append((A_poly, A_int, pct))
            if not kept_polys:
                # Keep the best-overlapping component instead of dropping the lake entirely
                best_g, best_attr = None, None
                best_A = -1.0
                for g in polys:
                    A_poly = geodesic_area_km2(g)
                    inter  = g.intersection(lake_unbuf)
                    A_int  = geodesic_area_km2(inter) if not inter.is_empty else 0.0
                    if A_int > best_A:
                        best_A = A_int
                        best_g = g
                        best_attr = (A_poly, A_int, (100.0 * A_int / A_poly) if A_poly > 0 else 0.0)
                if best_g is not None:
                    kept_polys = [best_g]
                    kept_attrs = [best_attr]

            # Keep only top-N by overlap area if requested
            if KEEP_TOP_N_BY_OVERLAP is not None and len(kept_polys) > KEEP_TOP_N_BY_OVERLAP:
                ranked = sorted(zip(kept_polys, kept_attrs), key=lambda t: t[1][1], reverse=True)
                ranked = ranked[:KEEP_TOP_N_BY_OVERLAP]
                kept_polys, kept_attrs = zip(*ranked)
                kept_polys, kept_attrs = list(kept_polys), list(kept_attrs)

            polys = kept_polys
            attrs = kept_attrs
        else:
            # No overlap filtering; compute areas only
            attrs = [(geodesic_area_km2(g), np.nan, np.nan) for g in polys]
        # -------------------------------------------------------------------------

        # Dissolve option
        if dissolve:
            geom_diss = unary_union(polys)
            # Apply hole removal again after dissolve (safer)
            if FILL_SMALL_HOLES and MAX_HOLE_AREA_KM2 > 0:
                geom_diss = remove_small_holes(geom_diss, MAX_HOLE_AREA_KM2)
            polys = [geom_diss] if (geom_diss and not geom_diss.is_empty) else []
            if not polys:
                return _write_empty()
            # Recompute attributes for dissolved geometry
            A_poly = geodesic_area_km2(polys[0])
            if MODE.upper() == "BY_LAKE" and lake_name and KEEP_ONLY_AOI_OVERLAP:
                inter = polys[0].intersection(AOIs_WGS84[lake_name])
                A_int = geodesic_area_km2(inter) if not inter.is_empty else 0.0
                pct   = (100.0 * A_int / A_poly) if A_poly > 0 else 0.0
            else:
                A_int, pct = (np.nan, np.nan)
            attrs = [(A_poly, A_int, pct)]

        # Build attributes with SHP-safe column names
        records = []
        for (A_poly, A_int, pct), g in zip(attrs, polys):
            rec = {
                "year":  int(yyyy),
                "month": int(mm),
                "lake":  (lake_name or "full"),
                COL_AREA: float(A_poly)
            }
            if MODE.upper() == "BY_LAKE" and lake_name and KEEP_ONLY_AOI_OVERLAP:
                rec[COL_OVA] = float(A_int)
                rec[COL_OVP] = float(pct)
            records.append(rec)

        gdf = gpd.GeoDataFrame(records, geometry=polys, crs="EPSG:4326")

        out_dir.mkdir(parents=True, exist_ok=True)
        gdf.to_file(out_path, driver=drv)

    return out_path


In [7]:
# ==============================================================
# 7) Driver — iterate mosaics and generate vectors (single-run supported)
# ==============================================================

# Select mosaics (supports single-file test via RUN_ONLY_YYYY_MM)
if RUN_ONLY_YYYY_MM is None:
    mosaics = sorted(MOSAIC_DIR.glob("sw_????_??.tif"))
    print(f"Found {len(mosaics)} monthly mosaics in {MOSAIC_DIR}")
else:
    m = re.match(r"^\d{4}_\d{2}$", RUN_ONLY_YYYY_MM)
    if not m:
        raise ValueError("RUN_ONLY_YYYY_MM must be like '2024_07' or None.")
    target = MOSAIC_DIR / f"sw_{RUN_ONLY_YYYY_MM}.tif"
    if not target.exists():
        raise FileNotFoundError(f"Requested single-run file not found: {target}")
    mosaics = [target]
    print(f"[single-run] Processing only: {target.name}")

written: List[Path] = []

if MODE.upper() == "BY_LAKE":
    targets = [
        ("Tanganyika", AOIs_BUFFERED["Tanganyika"], OUT_SUBDIR / "Tanganyika"),
        ("Kivu",       AOIs_BUFFERED["Kivu"],       OUT_SUBDIR / "Kivu")
    ]
else:
    targets = [(None, None, OUT_SUBDIR)]  # Full raster, no clipping

for m in tqdm(mosaics, desc="Vectorizing mosaics", unit="month"):
    for lake_name, aoi_geom, odir in targets:
        p = process_mosaic_for_target(
            m, lake_name, aoi_geom, odir,
            overwrite=OVERWRITE,
            dissolve=DISSOLVE_TO_SINGLE,
            drop_small_km2=MIN_POLY_AREA_KM2
        )
        if p is not None:
            written.append(p)

print(f"Completed: wrote {len(written)} vector file(s).")


Found 43 monthly mosaics in C:\Users\ibana\Desktop\JRC_Tanganica\GIS_Intermediate\Intermediate_files\SurfaceWater\GEE_SW\mosaics


Vectorizing mosaics:   0%|          | 0/43 [00:00<?, ?month/s]

[diag] sw_2022_01.tif / Tanganyika: water_px=331888004
[diag] sw_2022_01.tif / Kivu: water_px=24288104


Vectorizing mosaics:   2%|▏         | 1/43 [00:45<32:07, 45.90s/month]

[diag] sw_2022_02.tif / Tanganyika: water_px=331961259
[diag] sw_2022_02.tif / Kivu: water_px=24275370


...

[diag] sw_2025_06.tif / Tanganyika: water_px=332230861
[diag] sw_2025_06.tif / Kivu: water_px=24288600


Vectorizing mosaics:  98%|█████████▊| 42/43 [31:35<00:45, 45.00s/month]

[diag] sw_2025_07.tif / Tanganyika: water_px=332205461
[diag] sw_2025_07.tif / Kivu: water_px=24251190


Vectorizing mosaics: 100%|██████████| 43/43 [32:19<00:00, 45.11s/month]﻿


Completed: wrote 86 vector file(s).


In [8]:
# ==============================================================
# 8) Inventory — quick QA table of written vectors (area & parts)
# ==============================================================

rows = []
for p in written:
    m = re.search(r"sw_(\d{4})_(\d{2})_(Tanganyika|Kivu|full)\.(?:gpkg|shp)$",
                  p.name, re.IGNORECASE)
    year  = int(m.group(1)) if m else None
    month = int(m.group(2)) if m else None
    lake  = m.group(3) if m else None

    try:
        g = gpd.read_file(p)
        total_area = float(g["area_km2"].sum()) if "area_km2" in g.columns else np.nan
        n_parts    = len(g)
    except Exception:
        total_area = np.nan
        n_parts    = np.nan

    rows.append({
        "file": str(p),
        "year": year,
        "month": month,
        "lake": lake,
        "parts": n_parts,
        "area_km2_sum": total_area
    })

inv_df = pd.DataFrame(rows).sort_values(["lake","year","month"]).reset_index(drop=True)
print(inv_df.head(12).to_string(index=False))

# Save inventory next to GEE_SW vectors
inv_csv = OUT_SUBDIR / "gee_sw_vectors_inventory.csv"
inv_df.to_csv(inv_csv, index=False, encoding="utf-8-sig")
print("Inventory saved →", inv_csv)


                                                                                                                         file  year  month lake  parts  area_km2_sum
C:\Users\ibana\Desktop\JRC_Tanganica\GIS_Intermediate\Intermediate_files\SurfaceWater\vectors\GEE_SW\Kivu\sw_2022_01_Kivu.shp  2022      1 Kivu      1   2420.497452
C:\Users\ibana\Desktop\JRC_Tanganica\GIS_Intermediate\Intermediate_files\SurfaceWater\vectors\GEE_SW\Kivu\sw_2022_02_Kivu.shp  2022      2 Kivu      1   2418.396240
C:\Users\ibana\Desktop\JRC_Tanganica\GIS_Intermediate\Intermediate_files\SurfaceWater\vectors\GEE_SW\Kivu\sw_2022_03_Kivu.shp  2022      3 Kivu      1   2420.319727
C:\Users\ibana\Desktop\JRC_Tanganica\GIS_Intermediate\Intermediate_files\SurfaceWater\vectors\GEE_SW\Kivu\sw_2022_04_Kivu.shp  2022      4 Kivu      1   2419.995521
C:\Users\ibana\Desktop\JRC_Tanganica\GIS_Intermediate\Intermediate_files\SurfaceWater\vectors\GEE_SW\Kivu\sw_2022_05_Kivu.shp  2022      5 Kivu      1   2420.308465
C:\Users\i